Automated Medallion Pipeline – Single Execution Flow

This notebook executes a full Medallion architecture pipeline:

Bronze (Raw Ingestion)
→ Silver (Cleansed & Typed)
→ Gold (Business KPI Aggregation)

All stages execute in a single run without manual intervention.

In [55]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count
from pyspark.sql.types import IntegerType, DoubleType

spark = SparkSession.builder.appName("InstacartMedallionPipeline").getOrCreate()

base_path = "/content/instacart_pipeline"
bronze_path = f"{base_path}/bronze"
silver_path = f"{base_path}/silver"
gold_path = f"{base_path}/gold"

os.makedirs(bronze_path, exist_ok=True)
os.makedirs(silver_path, exist_ok=True)
os.makedirs(gold_path, exist_ok=True)

In [56]:
orders_df = spark.read.csv("/content/orders.csv", header=True)
orders_df.write.mode("overwrite").option("header", True).csv(bronze_path + "/orders_raw")

products_df = spark.read.csv("/content/products.csv", header=True)
products_df.write.mode("overwrite").option("header", True).csv(bronze_path + "/products_raw")

departments_df = spark.read.csv("/content/departments.csv", header=True)
departments_df.write.mode("overwrite").option("header", True).csv(bronze_path + "/departments_raw")

order_products_df = spark.read.csv("/content/order_products__prior.csv", header=True)
order_products_df.write.mode("overwrite").option("header", True).csv(bronze_path + "/order_products_raw")

In [57]:
orders_bronze = spark.read.csv(bronze_path + "/orders_raw", header=True)

orders_silver = orders_bronze \
    .withColumn("order_id", col("order_id").cast(IntegerType())) \
    .withColumn("user_id", col("user_id").cast(IntegerType())) \
    .withColumn("order_number", col("order_number").cast(IntegerType())) \
    .withColumn("order_dow", col("order_dow").cast(IntegerType())) \
    .withColumn("order_hour_of_day", col("order_hour_of_day").cast(IntegerType())) \
    .withColumn("days_since_prior_order",
                col("days_since_prior_order").cast(DoubleType()).cast(IntegerType()))

orders_silver.write.mode("overwrite").parquet(silver_path + "/orders_clean")

In [58]:
products = spark.read.csv(bronze_path + "/products_raw", header=True)
departments = spark.read.csv(bronze_path + "/departments_raw", header=True)
order_products = spark.read.csv(bronze_path + "/order_products_raw", header=True)

gold_table = order_products \
    .join(products, "product_id") \
    .join(departments, "department_id") \
    .groupBy("department") \
    .agg(count("order_id").alias("total_orders")) \
    .orderBy("total_orders", ascending=False)

gold_table.write.mode("overwrite").parquet(gold_path + "/department_kpi")

In [59]:
final_gold = spark.read.parquet(gold_path + "/department_kpi")
final_gold.show()

+---------------+------------+
|     department|total_orders|
+---------------+------------+
|        produce|     9479291|
|     dairy eggs|     5414016|
|         snacks|     2887550|
|      beverages|     2690129|
|         frozen|     2236432|
|         pantry|     1875577|
|         bakery|     1176787|
|   canned goods|     1068058|
|           deli|     1051249|
|dry goods pasta|      866627|
|      household|      738663|
|      breakfast|      709569|
|   meat seafood|      708931|
|  personal care|      447123|
|         babies|      423802|
|  international|      269253|
|        alcohol|      153696|
|           pets|       97724|
|        missing|       69145|
|          other|       36291|
+---------------+------------+
only showing top 20 rows
